# Alternative Regional Pressure (ExtraTrees)

Este notebook treina o modelo do serviço `alternative-regional-pressure` com saída compatível com o `regional-mobility-pressure`.

Entradas:
- `traffic-flow`
- `bus-regional-supply`

Saídas:
- classe: `baixa`, `media`, `alta`
- probabilidades de classe (`predict_proba`) para cálculo de `pressure_index` no serviço


In [8]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TRAFFIC_PATH = Path('../../L0/traffic-flow/output-example.txt').resolve()
BUS_SUPPLY_PATH = Path('../../L0/bus-regional-supply/output-example.txt').resolve()
MODEL_PATH = Path('model.joblib').resolve()

TRAFFIC_PATH, BUS_SUPPLY_PATH, MODEL_PATH


(PosixPath('/home/makleyston/Projects/service-composition/services/L0/traffic-flow/output-example.txt'),
 PosixPath('/home/makleyston/Projects/service-composition/services/L0/bus-regional-supply/output-example.txt'),
 PosixPath('/home/makleyston/Projects/service-composition/services/L1/alternative-regional-pressure/model.joblib'))

In [9]:
def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def parse_traffic(records: list[dict]) -> pd.DataFrame:
    rows = []
    for rec in records:
        payload = rec.get('urn:ufcity:payload', {})
        features = payload.get('features', {})
        result = rec.get('saref:hasResult', {})
        rows.append(
            {
                'traffic_level': result.get('saref:hasValue'),
                'traffic_confidence': result.get('urn:ufcity:confidence'),
                'traffic_direction': features.get('SENTIDO'),
                'traffic_lane': features.get('FAIXA'),
                'vehicle_count': features.get('vehicle_count'),
                'avg_speed': features.get('avg_speed'),
                'speed_std': features.get('speed_std'),
                'pct_moto': features.get('pct_moto'),
                'pct_heavy': features.get('pct_heavy'),
                'traffic_hour': features.get('hour'),
                'traffic_weekday': features.get('weekday'),
            }
        )
    return pd.DataFrame(rows)


def parse_bus_supply(records: list[dict]) -> pd.DataFrame:
    rows = []
    for rec in records:
        payload = rec.get('urn:ufcity:payload', {})
        result = rec.get('saref:hasResult', {})
        hr = str(payload.get('HR', ''))
        hour = int(hr[8:10]) if len(hr) >= 10 and hr.isdigit() else np.nan
        weekday = (
            pd.to_datetime(hr, format='%Y%m%d%H%M%S', errors='coerce').dayofweek
            if len(hr) == 14 and hr.isdigit()
            else np.nan
        )
        rows.append(
            {
                'bus_supply_level': result.get('saref:hasValue'),
                'bus_supply_confidence': result.get('urn:ufcity:confidence'),
                'bus_lt': payload.get('LT'),
                'bus_lg': payload.get('LG'),
                'bus_vl': payload.get('VL'),
                'bus_nl': payload.get('NL'),
                'bus_dg': payload.get('DG'),
                'bus_sv': payload.get('SV'),
                'bus_dt': payload.get('DT'),
                'bus_hour': hour,
                'bus_weekday': weekday,
            }
        )
    return pd.DataFrame(rows)


traffic_df = parse_traffic(load_jsonl(TRAFFIC_PATH))
bus_df = parse_bus_supply(load_jsonl(BUS_SUPPLY_PATH))

traffic_df.shape, bus_df.shape


((5819, 11), (3282, 11))

In [10]:
# Combinação sintética para PoC (mesma estratégia do serviço original)
sample_size = min(len(traffic_df), len(bus_df))
traffic_sample = traffic_df.sample(n=sample_size, random_state=42, replace=False).reset_index(drop=True)
bus_sample = bus_df.sample(n=sample_size, random_state=42, replace=False).reset_index(drop=True)
df = pd.concat([traffic_sample, bus_sample], axis=1)

numeric_cols = [
    'traffic_confidence', 'traffic_lane', 'vehicle_count', 'avg_speed', 'speed_std',
    'pct_moto', 'pct_heavy', 'traffic_hour', 'traffic_weekday',
    'bus_supply_confidence', 'bus_lt', 'bus_lg', 'bus_vl', 'bus_nl',
    'bus_dg', 'bus_sv', 'bus_dt', 'bus_hour', 'bus_weekday'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

traffic_map = {'baixo': 20, 'moderado': 60, 'alto': 90}
supply_map = {'alta': 20, 'media': 50, 'baixa': 80}
traffic_score = df['traffic_level'].map(traffic_map).fillna(50)
supply_score = df['bus_supply_level'].map(supply_map).fillna(50)
speed_penalty = np.clip((50 - df['avg_speed'].fillna(30)) * 2, 0, 100)
delay_penalty = np.clip((df['bus_dt'].fillna(0) / 20000) * 100, 0, 100)

rule_score = (0.45 * traffic_score + 0.25 * supply_score + 0.20 * speed_penalty + 0.10 * delay_penalty) * 1.7
df['pressure_rule_score'] = np.clip(rule_score, 0, 100)
df['regional_mobility_pressure'] = pd.cut(
    df['pressure_rule_score'],
    bins=[-1, 40, 70, 101],
    labels=['baixa', 'media', 'alta'],
).astype('object')

df['regional_mobility_pressure'].value_counts(normalize=True)


regional_mobility_pressure
baixa    0.755027
media    0.230957
alta     0.014016
Name: proportion, dtype: float64

In [11]:
feature_cols = [
    'traffic_level', 'traffic_confidence', 'traffic_direction', 'traffic_lane', 'vehicle_count',
    'avg_speed', 'speed_std', 'pct_moto', 'pct_heavy', 'traffic_hour', 'traffic_weekday',
    'bus_supply_level', 'bus_supply_confidence', 'bus_lt', 'bus_lg', 'bus_vl', 'bus_nl',
    'bus_dg', 'bus_sv', 'bus_dt', 'bus_hour', 'bus_weekday'
]

numeric_features = [
    'traffic_confidence', 'traffic_lane', 'vehicle_count', 'avg_speed', 'speed_std', 'pct_moto', 'pct_heavy',
    'traffic_hour', 'traffic_weekday', 'bus_supply_confidence', 'bus_lt', 'bus_lg', 'bus_vl', 'bus_nl',
    'bus_dg', 'bus_sv', 'bus_dt', 'bus_hour', 'bus_weekday'
]
categorical_features = ['traffic_level', 'traffic_direction', 'bus_supply_level']

X = df[feature_cols]
y = df['regional_mobility_pressure']

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore')),
            ]),
            categorical_features,
        ),
    ]
)

clf = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        (
            'model',
            ExtraTreesClassifier(
                n_estimators=300,
                max_depth=18,
                min_samples_leaf=1,
                class_weight='balanced_subsample',
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

report = classification_report(y_test, y_pred, output_dict=True)
report['macro avg']['f1-score']


0.9761904761904763

In [12]:
index_weights = {'baixa': 0.0, 'media': 50.0, 'alta': 100.0}
artifact = {
    'pipeline': clf,
    'features': feature_cols,
    'index_weights': index_weights,
    'metadata': {
        'model_type': 'ExtraTreesClassifier',
        'service': 'alternative-regional-pressure',
        'random_state': 42,
    },
}

joblib.dump(artifact, MODEL_PATH)
MODEL_PATH


PosixPath('/home/makleyston/Projects/service-composition/services/L1/alternative-regional-pressure/model.joblib')